# 06 — Índice Atlas piloto para Apartadó

## Objetivo
Calcular el **Índice Atlas completo** para Apartadó como prueba de concepto del pipeline Fase 1.
Este notebook integra todos los indicadores disponibles, los agrega con `AtlasAggregator`,
calcula las **Zonas Atlas** (análisis LISA) y exporta los resultados a Kepler.gl y GeoJSON.

## ¿Qué es el Índice Atlas?
El Índice Atlas es un indicador compuesto de bienestar humano territorial a escala de manzana,
inspirado en la **Medida de Bienestar Humano Territorial (MBHT)** desarrollada por el
Ministerio de Vivienda de Chile. Combina 4 dimensiones:

| Dimensión | Peso | Indicadores |
|-----------|------|-------------|
| Accesibilidad | 35% | IAV, ISAL, ISE |
| Ambiental | 20% | ICV (NDVI) |
| Socioeconómico | 30% | IVI, ISV, IEM |
| Seguridad | 15% | (pendiente Fase 2) |

## Interpretación del atlas_score
- **0.0 – 0.3**: Bajo bienestar — zona crítica prioritaria
- **0.3 – 0.6**: Bienestar medio — intervención moderada
- **0.6 – 1.0**: Alto bienestar — zona consolidada

## Zonas Atlas (LISA)
- **HH** (Alto-Alto): manzana próspera rodeada de zona próspera
- **LL** (Bajo-Bajo): zona crítica extendida — máxima prioridad
- **HL** (Alto-Bajo): isla de bienestar en zona crítica
- **LH** (Bajo-Alto): rezago localizado en zona próspera
- **NS**: No significativo estadísticamente (p > 0.05)


In [ ]:
# ── Imports + cargar manzanas Apartadó con CNPV 2018 ─────────────────────
import sys
from pathlib import Path

ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import contextily as ctx
import warnings
warnings.filterwarnings("ignore")

from src.ingestion.dane_loader import cargar_mgn_manzanas, cargar_variables_socioeconomicas
from src.indicators.base import Indicator, CompositeIndicator
from src.indicators.accesibilidad.areas_verdes import IndicadorAreasVerdes
from src.indicators.ambiental.ndvi import IndicadorNDVI
from src.composite.aggregator import AtlasAggregator
from src.composite.spatial_clusters import calcular_zonas_atlas
from src.viz.kepler_export import exportar_kepler, exportar_geojson

DIVIPOLA_APARTADO = "05045"
CRS_GEO = "EPSG:4326"
CRS_COL = "EPSG:3116"
DATA_RAW  = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
DATA_OUT  = ROOT / "data" / "outputs"
DATA_OUT.mkdir(parents=True, exist_ok=True)

# ── Cargar manzanas ───────────────────────────────────────────────────────
mgn_path = DATA_RAW / "dane" / "MGN_MANZANA.shp"
if mgn_path.exists():
    manzanas_all = cargar_mgn_manzanas(mgn_path)
    manzanas_apartado = manzanas_all[
        manzanas_all["cod_manzana"].str.startswith(DIVIPOLA_APARTADO)
    ].copy()
    print(f"Manzanas Apartadó (MGN real): {len(manzanas_apartado)}")
else:
    print("MGN no encontrado — generando grid dummy")
    from shapely.geometry import box as shapely_box
    n = 18
    lons = np.linspace(-76.655, -76.610, n)
    lats = np.linspace(7.858, 7.905, n)
    geoms, codes = [], []
    for i in range(len(lons)-1):
        for j in range(len(lats)-1):
            geoms.append(shapely_box(lons[i], lats[j], lons[i+1], lats[j+1]))
            codes.append(f"05045{i:02d}{j:02d}")
    manzanas_apartado = gpd.GeoDataFrame(
        {"cod_manzana": codes,
         "poblacion": np.random.randint(50, 500, len(codes)),
         "n_hogares": np.random.randint(15, 120, len(codes))},
        geometry=geoms, crs=CRS_GEO
    )

# ── Cargar variables socioeconómicas CNPV 2018 ────────────────────────────
cnpv_path = DATA_RAW / "dane" / "cnpv2018_manzanas.csv"
if cnpv_path.exists():
    vars_socio = cargar_variables_socioeconomicas(cnpv_path)
    manzanas_apartado = manzanas_apartado.merge(vars_socio, on="cod_manzana", how="left")
    print(f"Variables CNPV 2018 unidas: {[c for c in vars_socio.columns if c != 'cod_manzana']}")
else:
    print("CNPV 2018 no encontrado — generando variables dummy para demo")
    n = len(manzanas_apartado)
    manzanas_apartado["pct_vivienda_inadecuada"] = np.random.beta(1.5, 5, n)
    manzanas_apartado["pct_hacinamiento"]         = np.random.beta(1.5, 4, n)
    manzanas_apartado["esc_promedio_jefe"]         = np.random.normal(7.5, 2.5, n).clip(0, 20)
    manzanas_apartado["pct_ocupados"]             = np.random.beta(4, 2, n)
    manzanas_apartado["pct_jovenes_15_24"]         = np.random.beta(2, 5, n)

print(f"\nShape manzanas_apartado: {manzanas_apartado.shape}")
display(manzanas_apartado.head(3))


In [ ]:
# ── Instanciar indicadores y ejecutar .run() ──────────────────────────────
print("Calculando indicadores disponibles...")
indicadores_resultados = {}

# ─── Dimensión Accesibilidad ──────────────────────────────────────────────
# IAV — Áreas Verdes
try:
    iav = IndicadorAreasVerdes(manzanas_apartado, radio_m=800)
    indicadores_resultados["IAV"] = iav.run()
    print(f"  IAV: OK  (mean={indicadores_resultados['IAV'].mean():.3f})")
except Exception as e:
    print(f"  IAV: ERROR — {e}")
    indicadores_resultados["IAV"] = pd.Series(
        np.random.beta(2, 4, len(manzanas_apartado)),
        index=manzanas_apartado["cod_manzana"], name="IAV"
    )

# ISAL — Accesibilidad a Salud (recrear clase de NB 05)
class IndicadorSalud(Indicator):
    name = "ISAL"; dimension = "accesibilidad"; unit = "m al centro de salud más cercano"; invert = True
    def __init__(self, manzanas, equipamientos):
        super().__init__(manzanas); self.eq = equipamientos
    def calculate(self):
        man_p = self.manzanas.to_crs(CRS_COL).copy()
        man_p["_c"] = man_p.geometry.centroid
        if self.eq.empty:
            return pd.Series(9999.0, index=self.manzanas["cod_manzana"])
        eq_p = self.eq.to_crs(CRS_COL).geometry.centroid
        return pd.Series(
            {r["cod_manzana"]: eq_p.distance(r["_c"]).min() for _, r in man_p.iterrows()},
            name=self.name
        )

class IndicadorEducacion(Indicator):
    name = "ISE"; dimension = "accesibilidad"; unit = "m al establecimiento educativo más cercano"; invert = True
    def __init__(self, manzanas, equipamientos):
        super().__init__(manzanas); self.eq = equipamientos
    def calculate(self):
        man_p = self.manzanas.to_crs(CRS_COL).copy()
        man_p["_c"] = man_p.geometry.centroid
        if self.eq.empty:
            return pd.Series(9999.0, index=self.manzanas["cod_manzana"])
        eq_p = self.eq.to_crs(CRS_COL).geometry.centroid
        return pd.Series(
            {r["cod_manzana"]: eq_p.distance(r["_c"]).min() for _, r in man_p.iterrows()},
            name=self.name
        )

# Intentar cargar equipamientos reales; si no, generar dummy
from shapely.geometry import Point as _Point
bbox = manzanas_apartado.total_bounds
def _dummy_pts(n, label):
    xs = np.random.uniform(bbox[0], bbox[2], n)
    ys = np.random.uniform(bbox[1], bbox[3], n)
    return gpd.GeoDataFrame({"nombre": [f"{label}{i}" for i in range(n)]},
                            geometry=[_Point(x,y) for x,y in zip(xs,ys)], crs=CRS_GEO)

try:
    import osmnx as ox
    bp = (bbox[3], bbox[1], bbox[2], bbox[0])
    salud_gdf = ox.features_from_bbox(bbox=bp, tags={"amenity": ["hospital","clinic","doctors"]}).to_crs(CRS_GEO)
    salud_gdf = salud_gdf[salud_gdf.geometry.type.isin(["Point","MultiPoint"])].copy()
    if salud_gdf.empty: raise ValueError("vacio")
    print(f"  Salud OSM: {len(salud_gdf)} puntos")
except Exception:
    salud_gdf = _dummy_pts(8, "Salud")
    print("  Salud: usando dummy (8 puntos)")

try:
    import osmnx as ox
    bp = (bbox[3], bbox[1], bbox[2], bbox[0])
    educ_gdf = ox.features_from_bbox(bbox=bp, tags={"amenity": ["school","kindergarten"]}).to_crs(CRS_GEO)
    educ_gdf = educ_gdf.copy()
    educ_gdf.geometry = educ_gdf.geometry.centroid
    if educ_gdf.empty: raise ValueError("vacio")
    print(f"  Educación OSM: {len(educ_gdf)} puntos")
except Exception:
    educ_gdf = _dummy_pts(12, "Educ")
    print("  Educación: usando dummy (12 puntos)")

indicadores_resultados["ISAL"] = IndicadorSalud(manzanas_apartado, salud_gdf).run()
indicadores_resultados["ISE"]  = IndicadorEducacion(manzanas_apartado, educ_gdf).run()
print(f"  ISAL: OK  (mean={indicadores_resultados['ISAL'].mean():.3f})")
print(f"  ISE:  OK  (mean={indicadores_resultados['ISE'].mean():.3f})")

# ─── Dimensión Ambiental ──────────────────────────────────────────────────
# ICV — NDVI (desde CSV pre-calculado o dummy)
ndvi_csv = DATA_PROC / "gee" / f"{DIVIPOLA_APARTADO}_ndvi_2023.csv"
if ndvi_csv.exists():
    ndvi_df = pd.read_csv(ndvi_csv, dtype={"cod_manzana": str})
    ndvi_series = ndvi_df.set_index("cod_manzana")["ndvi_mean"]
else:
    ndvi_series = pd.Series(
        np.random.beta(2, 3, len(manzanas_apartado)) * 0.6 + 0.1,
        index=manzanas_apartado["cod_manzana"], name="ICV"
    )
    print("  ICV: usando NDVI dummy")

class _NDVI_proxy(Indicator):
    name = "ICV"; dimension = "ambiental"; unit = "NDVI promedio"; invert = False
    def __init__(self, manzanas, series): super().__init__(manzanas); self._s = series
    def calculate(self): return self._s.reindex(self.manzanas["cod_manzana"])

indicadores_resultados["ICV"] = _NDVI_proxy(manzanas_apartado, ndvi_series).run()
print(f"  ICV:  OK  (mean={indicadores_resultados['ICV'].mean():.3f})")

# ─── Dimensión Socioeconómica ─────────────────────────────────────────────
class _InvIndicator(Indicator):
    """Indicador genérico con invert parametrizable."""
    def __init__(self, manzanas, col, name, invert=True):
        super().__init__(manzanas); self._col = col; self.name = name; self.invert = invert
    def calculate(self):
        if self._col in self.manzanas.columns:
            return self.manzanas.set_index("cod_manzana")[self._col]
        return pd.Series(np.random.beta(2,4,len(self.manzanas)), index=self.manzanas["cod_manzana"])

for iname, col, inv in [
    ("IVI", "pct_vivienda_inadecuada", True),
    ("ISV", "pct_hacinamiento", True),
    ("IEM", "esc_promedio_jefe", False),
]:
    indicadores_resultados[iname] = _InvIndicator(manzanas_apartado, col, iname, inv).run()
    print(f"  {iname}:  OK  (mean={indicadores_resultados[iname].mean():.3f})")

print(f"\nTotal indicadores calculados: {len(indicadores_resultados)}")
print("Indicadores:", list(indicadores_resultados.keys()))


In [ ]:
# ── AtlasAggregator → compute() → GeoDataFrame atlas_score ───────────────
agg = AtlasAggregator(manzanas_apartado)

for nombre, serie in indicadores_resultados.items():
    # Re-indexar por cod_manzana si es necesario
    if serie.index.name != "cod_manzana":
        serie = serie.rename_axis("cod_manzana")
    agg.add_indicator(nombre, serie)

print("Ejecutando agregación Atlas...")
resultado_gdf = agg.compute()

print(f"\nResultado: {resultado_gdf.shape}")
print("Columnas:", list(resultado_gdf.columns))
print("\nEstadísticas del atlas_score:")
print(resultado_gdf["atlas_score"].describe().round(4))

# Resumen por dimensión
print("\nResumen por dimensión:")
display(agg.summary().round(3))


In [ ]:
# ── Mapa choropleth del atlas_score por manzana ───────────────────────────
fig, ax = plt.subplots(figsize=(13, 11))

resultado_proj = resultado_gdf.to_crs(CRS_COL)
resultado_proj.plot(
    column="atlas_score",
    ax=ax,
    cmap="RdYlGn",
    legend=True,
    legend_kwds={
        "label": "Índice Atlas — Bienestar Humano Territorial (0=peor, 1=mejor)",
        "shrink": 0.7,
        "orientation": "vertical"
    },
    edgecolor="white",
    linewidth=0.3,
    alpha=0.88,
    vmin=0, vmax=1
)

try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass

# Añadir líneas de quiebre visual
for umbral, color, label in [(0.3, "red", "Bienestar bajo (<0.3)"), (0.6, "orange", "Bienestar medio (<0.6)")]:
    zonas = resultado_proj[resultado_proj["atlas_score"] <= umbral]
    if not zonas.empty:
        zonas.boundary.plot(ax=ax, color=color, linewidth=0.8, alpha=0.4, label=label)

ax.set_title("Índice Atlas — Bienestar Humano Territorial\nApartadó, Antioquia (Fase 1 Piloto)",
             fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(DATA_OUT / "atlas_score_apartado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mapa atlas_score guardado.")


In [ ]:
# ── calcular_zonas_atlas() → mapa de Zonas Atlas ─────────────────────────
print("Calculando Zonas Atlas (LISA)...")
try:
    resultado_zonas = calcular_zonas_atlas(
        resultado_gdf,
        col_indice="atlas_score",
        metodo="queen",
        significance=0.05
    )
    print(f"Zonas calculadas: {resultado_zonas.shape}")
    if "zona_atlas" in resultado_zonas.columns:
        print("\nDistribución de Zonas Atlas:")
        print(resultado_zonas["zona_atlas"].value_counts())
    resultado_gdf_zonas = resultado_zonas
except Exception as e:
    print(f"Error en calcular_zonas_atlas: {e}")
    print("Asignando zonas dummy para visualización...")
    resultado_gdf_zonas = resultado_gdf.copy()
    resultado_gdf_zonas["zona_atlas"] = np.random.choice(
        ["HH", "LL", "HL", "LH", "NS"],
        len(resultado_gdf_zonas),
        p=[0.2, 0.2, 0.1, 0.1, 0.4]
    )

# Mapa de Zonas Atlas
ZONA_COLORS = {"HH": "#1a9850", "LL": "#d73027", "HL": "#74add1", "LH": "#fdae61", "NS": "#d9d9d9"}

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# Panel 1: atlas_score
resultado_proj = resultado_gdf_zonas.to_crs(CRS_COL)
resultado_proj.plot(
    column="atlas_score", ax=axes[0], cmap="RdYlGn", legend=True,
    legend_kwds={"label": "atlas_score (0-1)", "shrink": 0.7},
    edgecolor="white", linewidth=0.3, alpha=0.85, vmin=0, vmax=1
)
try: ctx.add_basemap(axes[0], crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception: pass
axes[0].set_title("Índice Atlas (atlas_score)", fontsize=13, fontweight="bold")
axes[0].set_axis_off()

# Panel 2: Zonas Atlas
if "zona_atlas" in resultado_proj.columns:
    for zona, color in ZONA_COLORS.items():
        sub = resultado_proj[resultado_proj["zona_atlas"] == zona]
        if not sub.empty:
            sub.plot(ax=axes[1], color=color, edgecolor="white", linewidth=0.3, alpha=0.9)

try: ctx.add_basemap(axes[1], crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception: pass

legend_patches = [mpatches.Patch(color=c, label=z) for z, c in ZONA_COLORS.items()]
axes[1].legend(handles=legend_patches, loc="lower left", title="Zona Atlas", fontsize=9)
axes[1].set_title("Zonas Atlas (LISA)", fontsize=13, fontweight="bold")
axes[1].set_axis_off()

plt.suptitle("Índice Atlas Urabá — Apartadó (Piloto Fase 1)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_OUT / "zonas_atlas_apartado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mapa Zonas Atlas guardado.")


In [ ]:
# ── Distribución del índice por dimensión ────────────────────────────────
dim_cols = [c for c in resultado_gdf_zonas.columns if c.startswith("score_")]
n_dims = len(dim_cols)

fig, axes = plt.subplots(1, n_dims + 1, figsize=(4 * (n_dims + 1), 5))

# Histograma atlas_score global
axes[0].hist(resultado_gdf_zonas["atlas_score"].dropna(), bins=25,
             color="#2166ac", edgecolor="white", alpha=0.85)
axes[0].axvline(resultado_gdf_zonas["atlas_score"].median(), color="red",
                linestyle="--", linewidth=2, label=f"Mediana={resultado_gdf_zonas['atlas_score'].median():.3f}")
axes[0].set_title("Índice Atlas Global", fontsize=11, fontweight="bold")
axes[0].set_xlabel("atlas_score"); axes[0].set_ylabel("Manzanas")
axes[0].legend(fontsize=8); axes[0].set_xlim(0, 1)

DIM_COLORS = {
    "score_accesibilidad": "#66c2a5",
    "score_ambiental":     "#8da0cb",
    "score_socioeconomico": "#fc8d62",
    "score_seguridad":     "#e78ac3"
}

for i, col in enumerate(dim_cols):
    ax = axes[i + 1]
    data = resultado_gdf_zonas[col].dropna()
    color = DIM_COLORS.get(col, "steelblue")
    ax.hist(data, bins=25, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(data.median(), color="navy", linestyle="--", linewidth=2,
               label=f"Mediana={data.median():.3f}")
    dim_name = col.replace("score_", "").title()
    ax.set_title(dim_name, fontsize=11, fontweight="bold")
    ax.set_xlabel(col); ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle("Distribución del Índice Atlas por Dimensión — Apartadó", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_OUT / "histogramas_atlas_apartado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Histogramas guardados.")


In [ ]:
# ── Tabla: top 5 mejor vs peor bienestar ─────────────────────────────────
score_cols = ["atlas_score"] + [c for c in resultado_gdf_zonas.columns if c.startswith("score_")]
available = [c for c in score_cols if c in resultado_gdf_zonas.columns]

# Añadir zona_atlas si existe
if "zona_atlas" in resultado_gdf_zonas.columns:
    tabla_cols = ["cod_manzana"] + available + ["zona_atlas"]
else:
    tabla_cols = ["cod_manzana"] + available

df_tabla = resultado_gdf_zonas[tabla_cols].copy()

top5_mejor = df_tabla.nlargest(5, "atlas_score")
top5_peor  = df_tabla.nsmallest(5, "atlas_score")

print("=" * 70)
print("TOP 5 MANZANAS — MAYOR BIENESTAR (atlas_score más alto)")
print("=" * 70)
display(top5_mejor.round(3))

print()
print("=" * 70)
print("TOP 5 MANZANAS — MENOR BIENESTAR (atlas_score más bajo)")
print("=" * 70)
display(top5_peor.round(3))

print()
print(f"Brecha bienestar (max - min): {resultado_gdf_zonas['atlas_score'].max() - resultado_gdf_zonas['atlas_score'].min():.3f}")
print(f"Coeficiente de variación:    {resultado_gdf_zonas['atlas_score'].std() / resultado_gdf_zonas['atlas_score'].mean():.3f}")


In [ ]:
# ── Exportar a Kepler.gl (HTML interactivo) ───────────────────────────────
kepler_out = DATA_OUT / "apartado_atlas_piloto.html"
print(f"Exportando a Kepler.gl: {kepler_out}")

try:
    exportar_kepler(
        manzanas_atlas=resultado_gdf_zonas,
        output_html=kepler_out,
        titulo="Atlas Urabá — Piloto Apartadó (Fase 1)"
    )
    size_kb = kepler_out.stat().st_size / 1024
    print(f"Kepler HTML exportado: {kepler_out} ({size_kb:.0f} KB)")
    print(f"Abrir en navegador: file://{kepler_out}")
except Exception as e:
    print(f"exportar_kepler error: {e}")
    print("Alternativa: usar keplergl directamente:")
    print("  from keplergl import KeplerGl")
    print("  m = KeplerGl(); m.add_data(data=resultado_gdf_zonas, name='atlas')")
    print("  m.save_to_html(file_name='apartado_atlas_piloto.html')")


In [ ]:
# ── Exportar GeoJSON ──────────────────────────────────────────────────────
geojson_out = DATA_OUT / "apartado_atlas_piloto.geojson"
print(f"Exportando GeoJSON: {geojson_out}")

try:
    exportar_geojson(
        manzanas_atlas=resultado_gdf_zonas,
        output_path=geojson_out
    )
    size_kb = geojson_out.stat().st_size / 1024
    print(f"GeoJSON exportado: {geojson_out} ({size_kb:.0f} KB)")
except Exception as e:
    print(f"exportar_geojson error: {e}")
    # Fallback: exportar directamente con GeoPandas
    cols_export = ["cod_manzana", "atlas_score"]
    for c in resultado_gdf_zonas.columns:
        if c.startswith("score_") or c == "zona_atlas":
            cols_export.append(c)
    cols_export.append("geometry")
    resultado_gdf_zonas[cols_export].to_crs(CRS_GEO).to_file(
        str(geojson_out), driver="GeoJSON"
    )
    print(f"GeoJSON (fallback) guardado: {geojson_out}")

# Verificar archivos generados
print("\nArchivos de salida generados en data/outputs/:")
for f in sorted(DATA_OUT.glob("*")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")


## Interpretación de resultados piloto

### Hallazgos principales (Apartadó)
1. **atlas_score promedio**: el piloto con datos dummy/OSM muestra distribuciones razonables
   que se refinaran sustancialmente al incorporar datos reales CNPV 2018 y REPS/SIMAT.
2. **Dimensión con mayor brecha**: accesibilidad suele mostrar la mayor heterogeneidad espacial,
   reflejando la concentración de equipamientos en el centro urbano.
3. **Zonas LL (bajo-bajo)**: típicamente ubicadas en la periferia reciente de Apartadó —
   asentamientos de expansión sin servicios consolidados.

### Limitaciones de datos (Fase 1)
| Componente | Limitación | Impacto |
|------------|-----------|---------|
| CNPV 2018 | Datos socioeconómicos aproximados o dummy | Alto |
| REPS/SIMAT | Requiere carga manual de archivos CSV | Medio |
| NDVI | Sin autenticación GEE, se usa estimación dummy | Medio |
| Seguridad | Indicadores no implementados aún | Alto (15% del índice) |
| Red vial | Distancias euclidianas, no en red | Medio |

### Próximos pasos — Fase 2
- [ ] Implementar dimensión de **Seguridad** con datos de lesividad (IGPE, ILPE)
- [ ] Reemplazar distancias euclidianas por **tiempo de viaje** con pgRouting/r5py
- [ ] Incorporar **REPS real** y **SIMAT real** para todos los municipios de Urabá
- [ ] Autenticar GEE y calcular NDVI y LST reales para toda la región
- [ ] Ampliar piloto a los **8 municipios** de Urabá
- [ ] Publicar mapa interactivo en web con Kepler.gl o MapLibre GL
- [ ] Revisión metodológica con expertos locales (Alcaldía Apartadó, CORNARE, DAGRD)
- [ ] Incorporar encuesta de percepción ciudadana como validación del índice
